# Group 6 — Spark SQL Advanced Queries
**ITCS 6190/8190 Cloud Computing for Data Analysis**

Six complex SQL queries using window functions, CTEs, LATERAL VIEW EXPLODE, and multi-table JOINs.

## Setup & Register Views
Loads the 8 Parquet tables produced by `ingestion.ipynb` (S3 if `S3_BUCKET_PATH` is set, else local `data/processed/`) and registers them as Spark SQL temp views.

In [ ]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast
os.environ.pop('JAVA_TOOL_OPTIONS', None)
spark = SparkSession.builder \
    .appName('Group6-SQL') \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", os.environ.get("AWS_ACCESS_KEY_ID", "")) \
    .config("spark.hadoop.fs.s3a.secret.key", os.environ.get("AWS_SECRET_ACCESS_KEY", "")) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

BUCKET = os.environ.get("S3_BUCKET_PATH", "")
PROC = f"{BUCKET}/processed" if BUCKET else "../data/processed"

customers    = spark.read.parquet(f'{PROC}/customers_clean')
products     = spark.read.parquet(f'{PROC}/products_clean')
transactions = spark.read.parquet(f'{PROC}/transactions_exploded')
clicks       = spark.read.parquet(f'{PROC}/clickstream_clean')
sessions     = spark.read.parquet(f'{PROC}/session_funnel')

products_b = broadcast(products)

customers.createOrReplaceTempView('customers')
products_b.createOrReplaceTempView('products')
transactions.createOrReplaceTempView('transactions')
clicks.createOrReplaceTempView('clickstream')
sessions.createOrReplaceTempView('sessions')

print('Views registered: customers, products, transactions, clickstream, sessions')

Views registered: customers, products, transactions, clickstream, sessions


## Q1 — Per-Customer Conversion Funnel (4-table JOIN + window functions)
**Business question:** where does each customer drop off in the purchase journey, and how does engagement rank across the base?

**Techniques:** 4-table JOIN (clickstream → sessions → transactions → customers), 2 CTEs, `ROW_NUMBER`, `DENSE_RANK`, CASE segmentation.

In [3]:
spark.sql("""
WITH customer_sessions AS (
  SELECT
    c.customer_id,
    c.home_country,
    c.device_type,
    c.gender,
    s.session_id,
    s.visited_homepage,
    s.did_search,
    s.added_to_cart,
    s.used_promo,
    s.converted,
    s.session_duration_mins,
    ROW_NUMBER() OVER (
      PARTITION BY c.customer_id
      ORDER BY s.session_start
    ) AS session_seq
  FROM customers c
  JOIN transactions t ON t.customer_id = c.customer_id
  JOIN sessions     s ON s.session_id  = t.session_id
),
customer_funnel AS (
  SELECT
    customer_id,
    home_country,
    device_type,
    gender,
    COUNT(DISTINCT session_id)                AS sessions,
    MAX(session_seq)                          AS journey_depth,
    MAX(visited_homepage)                     AS hit_homepage,
    MAX(did_search)                           AS did_search,
    MAX(added_to_cart)                        AS added_to_cart,
    MAX(used_promo)                           AS used_promo,
    MAX(converted)                            AS converted,
    ROUND(AVG(session_duration_mins), 1)      AS avg_session_mins,
    (MAX(visited_homepage) + MAX(did_search)
     + MAX(added_to_cart) + MAX(converted))   AS funnel_score
  FROM customer_sessions
  GROUP BY customer_id, home_country, device_type, gender
)
SELECT *,
  CASE
    WHEN funnel_score = 4                     THEN 'CONVERTED'
    WHEN added_to_cart = 1 AND converted = 0  THEN 'HIGH_INTENT_DROPPED'
    WHEN did_search    = 1                    THEN 'MID_FUNNEL'
    ELSE 'TOP_FUNNEL_ONLY'
  END AS funnel_stage,
  DENSE_RANK() OVER (
    ORDER BY funnel_score DESC, sessions DESC
  ) AS engagement_rank
FROM customer_funnel
ORDER BY engagement_rank
""").show(20, truncate=False)

26/04/20 13:12:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 13:12:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 13:12:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 13:12:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 13:12:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 13:12:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 1

+-----------+------------+-----------+------+--------+-------------+------------+----------+-------------+----------+---------+----------------+------------+------------+---------------+
|customer_id|home_country|device_type|gender|sessions|journey_depth|hit_homepage|did_search|added_to_cart|used_promo|converted|avg_session_mins|funnel_score|funnel_stage|engagement_rank|
+-----------+------------+-----------+------+--------+-------------+------------+----------+-------------+----------+---------+----------------+------------+------------+---------------+
|43202      |Indonesia   |Android    |Women |550     |807          |1           |1         |1            |1         |1        |2199.0          |4           |CONVERTED   |1              |
|29496      |Indonesia   |Android    |Women |505     |749          |1           |1         |1            |1         |1        |1665.4          |4           |CONVERTED   |2              |
|82237      |Indonesia   |Android    |Women |503     |716        

## Q2 — Market Basket Analysis (self-JOIN on exploded line items)
**Business question:** which product pairs are frequently bought in the same booking?

**Techniques:** self-JOIN on `transactions_exploded` (already one row per product per booking), 3 CTEs, `CROSS JOIN`, support + confidence metrics, category enrichment.

In [4]:
spark.sql("""
WITH paid_items AS (
  SELECT booking_id, customer_id, product_id
  FROM transactions
  WHERE payment_status = 'Success' AND product_id IS NOT NULL
),
order_stats AS (
  SELECT COUNT(DISTINCT booking_id) AS total_orders FROM paid_items
),
item_support AS (
  SELECT product_id, COUNT(DISTINCT booking_id) AS product_orders
  FROM paid_items
  GROUP BY product_id
),
co_purchases AS (
  SELECT
    a.product_id AS product_a,
    b.product_id AS product_b,
    COUNT(DISTINCT a.booking_id)  AS co_count,
    COUNT(DISTINCT a.customer_id) AS unique_buyers
  FROM paid_items a
  JOIN paid_items b
    ON a.booking_id  = b.booking_id
   AND a.product_id  < b.product_id
  GROUP BY a.product_id, b.product_id
  HAVING COUNT(DISTINCT a.booking_id) >= 2
)
SELECT
  p1.productDisplayName AS product_A,
  p1.masterCategory     AS category_A,
  p2.productDisplayName AS product_B,
  p2.masterCategory     AS category_B,
  cp.co_count,
  cp.unique_buyers,
  ROUND(cp.co_count * 100.0 / os.total_orders, 3) AS support_pct,
  ROUND(cp.co_count * 100.0 / s1.product_orders, 1) AS confidence_A_to_B,
  ROUND(cp.co_count * 100.0 / s2.product_orders, 1) AS confidence_B_to_A
FROM co_purchases cp
CROSS JOIN order_stats os
JOIN products p1 ON cp.product_a = p1.product_id
JOIN products p2 ON cp.product_b = p2.product_id
JOIN item_support s1 ON cp.product_a = s1.product_id
JOIN item_support s2 ON cp.product_b = s2.product_id
ORDER BY cp.co_count DESC, confidence_A_to_B DESC
LIMIT 20
""").show(20, truncate=False)

+-------------------------------------------------------------+-----------+-------------------------------------------------------------+-------------+--------+-------------+-----------+-----------------+-----------------+
|product_A                                                    |category_A |product_B                                                    |category_B   |co_count|unique_buyers|support_pct|confidence_A_to_B|confidence_B_to_A|
+-------------------------------------------------------------+-----------+-------------------------------------------------------------+-------------+--------+-------------+-----------+-----------------+-----------------+
|Jealous 21 Women's White Blue Shorts                         |Apparel    |Lino Perros Women Blue Zip Red Wallet                        |Accessories  |2       |2            |0.000      |13.3             |6.9              |
|Fastrack Women Silver Dial Watch NA6004SL01                  |Accessories|CASIO ENTICER Women Black Dial An

## Q3 — RFM Scoring with NTILE Quintiles
**Business question:** segment customers into actionable groups (Champions, At Risk, Hibernating) from Recency / Frequency / Monetary scores.

**Techniques:** 3 CTEs, `NTILE(5)` quintile ranking on three dimensions, composite RFM score, CASE-based segmentation, `DENSE_RANK`. Note: `customers_clean` dropped `first_name`/`last_name` as PII — segmentation uses `customer_id` + country/gender.

In [5]:
spark.sql("""
WITH date_ref AS (
  SELECT MAX(created_at) AS max_date FROM transactions
),
rfm_raw AS (
  SELECT
    t.customer_id,
    c.gender,
    c.home_country,
    c.device_type,
    DATEDIFF(dr.max_date, MAX(t.created_at))          AS recency_days,
    COUNT(DISTINCT t.booking_id)                      AS frequency,
    ROUND(SUM(t.quantity * t.item_price), 2)          AS monetary
  FROM transactions t
  CROSS JOIN date_ref dr
  JOIN customers c ON t.customer_id = c.customer_id
  WHERE t.payment_status = 'Success'
  GROUP BY t.customer_id, c.gender, c.home_country, c.device_type, dr.max_date
),
rfm_scored AS (
  SELECT *,
    NTILE(5) OVER (ORDER BY recency_days ASC)   AS r_score,
    NTILE(5) OVER (ORDER BY frequency DESC)     AS f_score,
    NTILE(5) OVER (ORDER BY monetary DESC)      AS m_score
  FROM rfm_raw
)
SELECT
  customer_id, gender, home_country, device_type,
  recency_days, frequency, monetary,
  r_score, f_score, m_score,
  (r_score + f_score + m_score) AS rfm_total,
  CASE
    WHEN (r_score + f_score + m_score) >= 13 THEN 'Champions'
    WHEN (r_score + f_score + m_score) >= 10 THEN 'Loyal Customers'
    WHEN (r_score + f_score + m_score) >= 7  THEN 'Potential Loyalists'
    WHEN (r_score + f_score + m_score) >= 4  THEN 'At Risk'
    ELSE 'Hibernating'
  END AS rfm_segment,
  DENSE_RANK() OVER (
    ORDER BY (r_score + f_score + m_score) DESC, monetary DESC
  ) AS overall_rank
FROM rfm_scored
ORDER BY overall_rank
LIMIT 25
""").show(25, truncate=False)

26/04/20 13:16:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 13:16:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 13:16:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 13:16:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 13:16:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 13:16:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/20 1

+-----------+------+------------+-----------+------------+---------+--------+-------+-------+-------+---------+-----------+------------+
|customer_id|gender|home_country|device_type|recency_days|frequency|monetary|r_score|f_score|m_score|rfm_total|rfm_segment|overall_rank|
+-----------+------+------------+-----------+------------+---------+--------+-------+-------+-------+---------+-----------+------------+
|14988      |Women |Indonesia   |Android    |621         |1        |554852.0|5      |5      |5      |15       |Champions  |1           |
|35606      |Women |Indonesia   |Android    |1238        |1        |554778.0|5      |5      |5      |15       |Champions  |2           |
|2044       |Women |Indonesia   |Android    |480         |1        |554314.0|5      |5      |5      |15       |Champions  |3           |
|59318      |Women |Indonesia   |iOS        |833         |1        |553703.0|5      |5      |5      |15       |Champions  |4           |
|91625      |Men   |Indonesia   |Android 

## Q4 — Cohort Retention Analysis
**Business question:** of customers whose first purchase was in month X, what fraction return in month X+1, X+3, X+6?

**Techniques:** 2 CTEs, `MONTHS_BETWEEN`, date arithmetic, pivot-style CASE aggregation, `NULLIF` to avoid divide-by-zero.

In [6]:
spark.sql("""
WITH first_purchase AS (
  SELECT
    customer_id,
    MIN(date_format(created_at, 'yyyy-MM')) AS cohort_month
  FROM transactions
  WHERE payment_status = 'Success'
  GROUP BY customer_id
),
monthly_activity AS (
  SELECT
    t.customer_id,
    fp.cohort_month,
    CAST(MONTHS_BETWEEN(
      TO_DATE(CONCAT(date_format(t.created_at, 'yyyy-MM'), '-01')),
      TO_DATE(CONCAT(fp.cohort_month, '-01'))
    ) AS INT) AS months_since_first
  FROM transactions t
  JOIN first_purchase fp ON t.customer_id = fp.customer_id
  WHERE t.payment_status = 'Success'
)
SELECT
  cohort_month,
  COUNT(DISTINCT CASE WHEN months_since_first = 0 THEN customer_id END) AS m0_cohort_size,
  COUNT(DISTINCT CASE WHEN months_since_first = 1 THEN customer_id END) AS m1_retained,
  COUNT(DISTINCT CASE WHEN months_since_first = 3 THEN customer_id END) AS m3_retained,
  COUNT(DISTINCT CASE WHEN months_since_first = 6 THEN customer_id END) AS m6_retained,
  COUNT(DISTINCT CASE WHEN months_since_first >= 12 THEN customer_id END) AS m12_plus,
  ROUND(
    COUNT(DISTINCT CASE WHEN months_since_first = 3 THEN customer_id END) * 100.0 /
    NULLIF(COUNT(DISTINCT CASE WHEN months_since_first = 0 THEN customer_id END), 0),
    1
  ) AS retention_3m_pct,
  ROUND(
    COUNT(DISTINCT CASE WHEN months_since_first = 6 THEN customer_id END) * 100.0 /
    NULLIF(COUNT(DISTINCT CASE WHEN months_since_first = 0 THEN customer_id END), 0),
    1
  ) AS retention_6m_pct
FROM monthly_activity
GROUP BY cohort_month
ORDER BY cohort_month
""").show(30, truncate=False)

+------------+--------------+-----------+-----------+-----------+--------+----------------+----------------+
|cohort_month|m0_cohort_size|m1_retained|m3_retained|m6_retained|m12_plus|retention_3m_pct|retention_6m_pct|
+------------+--------------+-----------+-----------+-----------+--------+----------------+----------------+
|2016-06     |1             |0          |0          |0          |1       |0.0             |0.0             |
|2016-07     |268           |43         |57         |73         |210     |21.3            |27.2            |
|2016-08     |423           |90         |126        |126        |359     |29.8            |29.8            |
|2016-09     |435           |81         |108        |124        |369     |24.8            |28.5            |
|2016-10     |512           |104        |141        |152        |444     |27.5            |29.7            |
|2016-11     |488           |99         |145        |152        |406     |29.7            |31.1            |
|2016-12     |280  

## Q5 — Purchase Velocity & Gap Analysis
**Business question:** how often does each customer come back, and is their spend trending up or down?

**Techniques:** 2 CTEs, `LAG` for previous purchase time and amount, `DATEDIFF`, aggregation with STDDEV, multi-metric CASE segmentation.

In [7]:
spark.sql("""
WITH booking_totals AS (
  SELECT
    customer_id,
    booking_id,
    MIN(created_at)                       AS booking_at,
    SUM(quantity * item_price)            AS booking_amount
  FROM transactions
  WHERE payment_status = 'Success'
  GROUP BY customer_id, booking_id
),
ordered_purchases AS (
  SELECT
    customer_id,
    booking_id,
    booking_at,
    booking_amount,
    LAG(booking_at)     OVER (PARTITION BY customer_id ORDER BY booking_at) AS prev_at,
    LAG(booking_amount) OVER (PARTITION BY customer_id ORDER BY booking_at) AS prev_amount,
    ROW_NUMBER()        OVER (PARTITION BY customer_id ORDER BY booking_at) AS purchase_num,
    COUNT(*)            OVER (PARTITION BY customer_id)                     AS total_purchases
  FROM booking_totals
),
purchase_gaps AS (
  SELECT
    customer_id,
    total_purchases,
    DATEDIFF(booking_at, prev_at)             AS days_between,
    ROUND(booking_amount - prev_amount, 2)    AS spend_change,
    booking_amount
  FROM ordered_purchases
  WHERE prev_at IS NOT NULL
)
SELECT
  customer_id,
  total_purchases,
  COUNT(*)                                     AS repeat_purchases,
  ROUND(AVG(days_between), 1)                  AS avg_days_between,
  MIN(days_between)                            AS min_gap_days,
  MAX(days_between)                            AS max_gap_days,
  ROUND(STDDEV(days_between), 1)               AS stddev_gap,
  ROUND(AVG(spend_change), 2)                  AS avg_spend_change,
  ROUND(SUM(booking_amount), 2)                AS total_lifetime_spend,
  CASE
    WHEN AVG(days_between) < 60  THEN 'FREQUENT'
    WHEN AVG(days_between) < 120 THEN 'REGULAR'
    WHEN AVG(days_between) < 200 THEN 'OCCASIONAL'
    ELSE 'CHURNING'
  END AS purchase_velocity,
  CASE
    WHEN AVG(spend_change) >  50 THEN 'SPENDING_UP'
    WHEN AVG(spend_change) < -50 THEN 'SPENDING_DOWN'
    ELSE 'STABLE'
  END AS spend_trend
FROM purchase_gaps
GROUP BY customer_id, total_purchases
HAVING COUNT(*) >= 2
ORDER BY avg_days_between ASC
LIMIT 25
""").show(25, truncate=False)

+-----------+---------------+----------------+----------------+------------+------------+----------+----------------+--------------------+-----------------+-------------+
|customer_id|total_purchases|repeat_purchases|avg_days_between|min_gap_days|max_gap_days|stddev_gap|avg_spend_change|total_lifetime_spend|purchase_velocity|spend_trend  |
+-----------+---------------+----------------+----------------+------------+------------+----------+----------------+--------------------+-----------------+-------------+
|44642      |17             |16              |0.0             |0           |0           |0.0       |236.19          |1.4925025E7         |FREQUENT         |SPENDING_UP  |
|565        |4              |3               |0.0             |0           |0           |0.0       |83817.0         |1216004.0           |FREQUENT         |SPENDING_UP  |
|34547      |14             |13              |0.0             |0           |0           |0.0       |-82513.31       |7085911.0           |FREQUEN

## Q6 — Browse-to-Buy Product Affinity (4-table correlation)
**Business question:** which browsed product categories most reliably lead to purchases in which categories?

**Techniques:** 4 CTEs, 4-table join (clickstream → transactions → products ×2), browse/buy cross-correlation, lift metric.

In [8]:
# Q6 can blow past 3GB driver on the full clickstream × transactions self-join.
# Sample 10% of clickstream events — plenty for a realistic lift estimate.
spark.sql("""
WITH browsed AS (
  SELECT DISTINCT
    t.customer_id,
    cs.cs_product_id AS browsed_pid
  FROM clickstream TABLESAMPLE (10 PERCENT) cs
  JOIN transactions t ON cs.session_id = t.session_id
  WHERE cs.event_name = 'ADD_TO_CART'
    AND cs.cs_product_id IS NOT NULL
),
purchased AS (
  SELECT DISTINCT
    customer_id,
    product_id AS bought_pid
  FROM transactions
  WHERE payment_status = 'Success' AND product_id IS NOT NULL
),
browse_buy_pairs AS (
  SELECT
    b.customer_id,
    b.browsed_pid,
    p.bought_pid
  FROM browsed b
  JOIN purchased p
    ON b.customer_id = p.customer_id
   AND b.browsed_pid != p.bought_pid
),
product_stats AS (
  SELECT bought_pid, COUNT(DISTINCT customer_id) AS total_buyers
  FROM purchased
  GROUP BY bought_pid
)
SELECT
  pb.masterCategory AS browsed_category,
  pb.subCategory    AS browsed_subcategory,
  pp.masterCategory AS bought_category,
  pp.subCategory    AS bought_subcategory,
  COUNT(DISTINCT bbp.customer_id) AS num_customers,
  ROUND(
    COUNT(DISTINCT bbp.customer_id) * 100.0 /
    NULLIF(ps.total_buyers, 0),
    1
  ) AS browse_to_buy_lift_pct
FROM browse_buy_pairs bbp
JOIN products pb ON bbp.browsed_pid = pb.product_id
JOIN products pp ON bbp.bought_pid  = pp.product_id
JOIN product_stats ps ON bbp.bought_pid = ps.bought_pid
GROUP BY pb.masterCategory, pb.subCategory, pp.masterCategory, pp.subCategory, ps.total_buyers
HAVING COUNT(DISTINCT bbp.customer_id) >= 3
ORDER BY num_customers DESC
LIMIT 25
""").show(25, truncate=False)

26/04/20 13:17:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/20 13:17:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/20 13:18:02 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/20 13:18:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/20 13:18:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+----------------+-------------------+---------------+------------------+-------------+----------------------+
|browsed_category|browsed_subcategory|bought_category|bought_subcategory|num_customers|browse_to_buy_lift_pct|
+----------------+-------------------+---------------+------------------+-------------+----------------------+
|Footwear        |Shoes              |Apparel        |Topwear           |11165        |39875.0               |
|Footwear        |Shoes              |Apparel        |Topwear           |11106        |41133.3               |
|Footwear        |Shoes              |Apparel        |Topwear           |10750        |37069.0               |
|Footwear        |Shoes              |Apparel        |Topwear           |10725        |41250.0               |
|Apparel         |Topwear            |Apparel        |Topwear           |10495        |37482.1               |
|Apparel         |Topwear            |Apparel        |Topwear           |10455        |38722.2               |
|

## Summary

| # | Query | Key Techniques |
|---|-------|----------------|
| 1 | Per-Customer Conversion Funnel | 4-table JOIN, 2 CTEs, ROW_NUMBER, DENSE_RANK |
| 2 | Market Basket Analysis | self-JOIN, 3 CTEs, CROSS JOIN, support + confidence |
| 3 | RFM Scoring | NTILE(5) quintiles, composite score, CASE segmentation |
| 4 | Cohort Retention | MONTHS_BETWEEN, pivot CASE, NULLIF |
| 5 | Purchase Velocity | LAG, DATEDIFF, STDDEV, multi-metric CASE |
| 6 | Browse-to-Buy Affinity | 4 CTEs, 4-table correlation, lift metric |

## Run the full SQL script
```bash
python ../src/transformations.py
```
Outputs saved to `outputs/sql_results/`

> Note: `src/transformations.py` still references the legacy sample-data schema and will need to be re-synced to the new Parquet outputs (`*_clean`, flat columns, dropped PII) before it runs end-to-end. The notebook above is the authoritative, working version.